# College Major vs Salary — Full EDA

## The question

Should you study Engineering or Philosophy? Computer Science or Psychology? The standard advice — *pick STEM* — is vague and, as this data will show, not always right.

This notebook uses a [Wall Street Journal / PayScale](https://www.payscale.com/) survey of **1.2 million Americans with bachelor's degrees** to answer that question with data. The dataset covers **50 undergraduate majors** grouped into three broad categories:

- **STEM** — Science, Technology, Engineering, Mathematics
- **Business** — Business, Commerce, and applied professional fields
- **HASS** — Humanities, Arts, and Social Sciences

## What we will discover

By the end of this notebook you will be able to answer:

1. Which major pays the most — at graduation and at mid-career?
2. Which major has the highest *ceiling* — top 10% earners?
3. Which major gives you the most *career growth* from start to mid-career?
4. Which major is the *safest bet* — lowest variance between high and low earners?
5. Do STEM, Business, and HASS graduates actually earn differently — and by how much?
6. Are there HASS majors that beat the average STEM starting salary?

## How this notebook is structured

The analysis follows a deliberate pipeline: load → inspect → clean → engineer features → answer questions → visualise → export. Each section builds on the one before it. By the end, the cleaned and enriched dataset is ready for further analysis, reporting, or visualisation in any tool.

**Run:** Kernel → Restart & Run All to execute from top to bottom.

---

### Sections
| # | Section | Key question answered |
|---|---|---|
| 1  | [Load](#section-1-load) | Is the file readable and complete? |
| 2  | [Inspection & Profiling](#section-2-inspection) | What shape is the data in? Any problems? |
| 3  | [Cleaning](#section-3-cleaning) | Remove junk rows and standardise column names |
| 4  | [Feature Engineering](#section-4-feature-engineering) | What derived metrics do we need? |
| 5  | [Selection & Filtering](#section-5-selection) | How do we slice the data for specific questions? |
| 6  | [Aggregation](#section-6-aggregation) | What do the summary statistics tell us? Stores `top_mid_row`, `bot_mid_row` |
| 7  | [GroupBy](#section-7-groupby) | How do degree categories compare? Stores `stem_avg_mid`, `hass_avg_mid` |
| 8  | [Sorting & Ranking](#section-8-sorting) | Which majors rank highest on each metric? Stores `low_risk_row`, `high_ceil_row`, `top_grow_row` |
| 9  | [Reshaping & Pivoting](#section-9-reshaping) | How do we change the data's shape for new views? |
| 10 | [Apply & Map](#section-10-apply) | What can't built-in methods do that we can? |
| 11 | [Display & Styling](#section-11-display) | How do we make output readable? |
| 12 | [Correlation Analysis](#section-12-correlation) | Which salary metrics predict each other? Stores `pearson_start_mid`, `stem_avg_start`, `hass_beating_stem` |
| 13 | [Export](#section-13-export) | How do we save results for reuse? |
| 14 | [Visualisations](#section-14-visualisations) | What do the charts reveal that tables can't? |
| 15 | [Key Findings](#section-15-key-findings) | What does the data actually say about major choice? |

## Key Terms

### Degree categories

| Term | Full name | Description |
|---|---|---|
| **STEM** | Science, Technology, Engineering, Mathematics | Technical and quantitative fields |
| **Business** | Business & Commerce | Applied professional and commercial fields |
| **HASS** | Humanities, Arts, and Social Sciences | Interpretive, creative, and social fields |

### Column names

The raw dataset uses long column names. This notebook renames them to short tokens on load.

| Shorthand | Original name | What it measures |
|---|---|---|
| `Major` | Undergraduate Major | Name of the degree |
| `Start` | Starting Median Salary | Median salary at career entry |
| `Mid` | Mid-Career Median Salary | Median salary ~10–15 years in |
| `P10` | Mid-Career 10th Percentile Salary | Lower bound — 10% of graduates earn below this |
| `P90` | Mid-Career 90th Percentile Salary | Upper bound — 10% of graduates earn above this |
| `Group` | Group | Degree category: STEM / Business / HASS |

### Derived columns (added during analysis)

| Column | Formula | What it captures |
|---|---|---|
| `Spread` | P90 − P10 | Salary range — proxy for earnings risk |
| `Growth` | (Mid − Start) / Start × 100 | Percentage salary increase by mid-career |
| `Safety` | Start / Spread | Predictability relative to variability |
| `Rank_Change` | Start_Rank − Mid_Rank | How much a major's salary rank shifts by mid-career |

### Statistical terms

| Term | Meaning |
|---|---|
| **Median** | The middle value — half of graduates earn above, half below. More robust than the mean when outliers are present. |
| **Percentile** | P10 means 10% earn below this figure; P90 means 10% earn above it. |
| **Pearson r** | Correlation coefficient from −1 to +1. Above 0.7 indicates a strong positive linear relationship. |
| **Spread** | A wide spread means high variability — top earners do very well, bottom earners do not. A narrow spread means predictable outcomes. |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
from pathlib import Path

PLOTS_DIR = Path('plots')
PLOTS_DIR.mkdir(exist_ok=True)
DATA_PATH = Path('data/salaries_by_college_major.csv')

GROUP_COLOURS = {'STEM': '#2E75B6', 'Business': '#ED7D31', 'HASS': '#70AD47'}

## <a id="section-1-load"></a>1. Load

`pd.read_csv()` is the standard entry point for tabular data — it parses the file, infers column types, and returns a DataFrame in one call. No configuration needed for well-formed CSV files.

We print `len(df)` immediately as a sanity check: if the number is wildly different from what we expect, the file path is wrong, the file is truncated, or there is an encoding issue. Catching this now prevents confusing errors later.

Notice we do **not** call `df.head()` here and declare the data ready — that comes in the next section after a proper profiling pass.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} rows from {DATA_PATH}')
df.head()

## <a id="section-2-inspection"></a>2. Inspection & Profiling

**Never trust data you haven't profiled.** Before writing a single line of analysis, we answer a checklist of structural questions:

| Method | What it tells us |
|---|---|
| `shape` | How many rows and columns |
| `info()` | Column names, dtypes, and non-null counts in one view |
| `describe()` | Distribution of numeric columns — mean, std, min, quartiles, max |
| `dtypes` | Whether pandas inferred the right type for each column |
| `nunique()` | Number of distinct values — flags columns that should be categorical |
| `value_counts()` | Frequency of each group — tells us how the three categories are sized |
| `duplicated().sum()` | Whether any row appears more than once |
| `memory_usage(deep=True)` | How much RAM the DataFrame consumes |
| `isna()` / `isna().sum()` | Where missing values are, and how many per column |
| `head()` / `tail()` | A visual sanity check at both ends of the data |

**What to look for:** `isna().sum()` and `tail()` together will reveal the key issue — there is a footer row at the bottom of the file (`Source: PayScale Inc.`) that carries NaN for every numeric column. It is metadata disguised as a data row, and it must be removed before any calculation.

Every aggregation, every sort, every groupby in the sections that follow relies on this profiling having been done correctly. Skip it and you will eventually get a wrong answer that looks right.

In [ ]:
print('shape:', df.shape)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('dtypes:')
print(df.dtypes)
print('\nnunique:')
print(df.nunique())

In [ ]:
print('Group value counts:')
print(df['Group'].value_counts())
print('\nDuplicated rows:', df.duplicated().sum())
print('\nmemory_usage (deep):')
print(df.memory_usage(deep=True))

In [ ]:
print('isna per column:')
print(df.isna().sum())

In [ ]:
# tail() reveals the NaN footer row — the source of the missing values above
df.tail()

## <a id="section-3-cleaning"></a>3. Cleaning

### The problem

`tail()` exposed it: the last row is not a major. It is a source attribution — `Source: PayScale Inc.` — inserted by whoever compiled the spreadsheet. Pandas read it in as a row like any other, so `df.shape` reports one extra row and all four numeric columns show one NaN.

If we leave this row in, every aggregation — every `.mean()`, `.max()`, `.groupby()` — either produces a wrong result or raises a warning. It has to go.

### Why `dropna()` is appropriate here

`dropna()` removes every row that contains at least one NaN. Here, the *only* NaN row is the footer, so this is safe. In a real dataset with genuine missing values scattered across rows, you would not use `dropna()` blindly — you would investigate whether to fill, interpolate, or drop on a column-by-column basis. Here the choice is clear.

### Why we rename the columns

The original column names are descriptive but verbose. Writing `clean_df['Mid-Career 10th Percentile Salary']` twenty times in analysis code is tedious and error-prone. Short tokens — `Start`, `Mid`, `P10`, `P90` — make every subsequent line faster to read and harder to mistype.

### The `fillna()` alternative

Sometimes you want to fill NaN rather than drop the row — for example, filling with the column mean when data is missing at random. The cell below demonstrates this on a copy so you can see the difference. We do **not** apply it to `clean_df` — filling `Source: PayScale Inc.` with `$0` would add a fake major with zero salary to every downstream analysis.

In [ ]:
clean_df = df.dropna()
# Reset so the index runs cleanly from 0 after the footer row is removed
clean_df = clean_df.reset_index(drop=True)
clean_df = clean_df.rename(columns={
    'Undergraduate Major':               'Major',
    'Starting Median Salary':            'Start',
    'Mid-Career Median Salary':          'Mid',
    'Mid-Career 10th Percentile Salary': 'P10',
    'Mid-Career 90th Percentile Salary': 'P90',
})
print(f'After cleaning: {len(clean_df)} majors, index {clean_df.index[0]}–{clean_df.index[-1]}')
clean_df.head()

In [ ]:
# fillna demo on a copy — shows the alternative approach without touching clean_df
fillna_demo = df.copy()
fillna_demo[['Starting Median Salary', 'Mid-Career Median Salary']] = (
    fillna_demo[['Starting Median Salary', 'Mid-Career Median Salary']].fillna(0)
)
print('fillna result (last 3 rows of copy — note the $0 salary row):')
fillna_demo.tail(3)

## <a id="section-4-feature-engineering"></a>4. Feature Engineering

The four salary columns in the raw data — `Start`, `Mid`, `P10`, `P90` — describe *salary levels*. But the most interesting questions are about *salary dynamics*: how risky is this major? How much do you grow? Are you above or below average for your degree type?

To answer those, we need to engineer new columns from the existing ones. This is one of the most important and underappreciated steps in any analysis.

### What each derived column captures

**`Spread` = P90 − P10**  
The difference between the top 10% and bottom 10% of mid-career earners in the same field. A small spread means the field is predictable — graduates earn within a narrow band. A large spread means high variability — the top earners do extremely well, but so do the worst. *Spread is a proxy for earnings risk.*

**`Growth` = (Mid − Start) / Start × 100**  
The percentage increase from starting to mid-career salary. Some majors start modestly but grow dramatically. Others start high and plateau. Growth separates the long-game majors from the immediate-payout ones.

**`Safety` = Start / Spread**  
A ratio that rewards both a high starting salary and a narrow spread. A major with high Safety gives you a solid floor relative to how unpredictable the field is. Nursing scores high; Philosophy scores low.

**`Start_Rank` and `Mid_Rank`**  
Each major's rank by salary (rank 1 = highest) at start and at mid-career. Using `ascending=False` ensures rank 1 is the top earner, not the bottom.

**`Rank_Change` = Start_Rank − Mid_Rank**  
How much a major's rank shifts between graduation and mid-career. A *positive* Rank_Change means the major climbed — it started lower in the rankings but rose by mid-career (e.g. Philosophy). A *negative* Rank_Change means it fell (e.g. Physician Assistant starts very high but is surpassed by mid-career).

**`Salary_Band`**  
Equal-width cut of starting salary into Low / Medium / High. Useful for categorical grouping and colour-coding.

**`Group_Avg_Start` and `vs_group_avg`**  
`transform('mean')` computes the per-group average but returns a Series the same length as the DataFrame, so we can add it directly as a column. `vs_group_avg` then shows how far above or below the group average each individual major sits. A Philosophy major beating the HASS average is a very different thing from beating the STEM average.

In [ ]:
clean_df['Spread']   = clean_df['P90'] - clean_df['P10']
clean_df['Growth']   = (clean_df['Mid'] - clean_df['Start']) / clean_df['Start'] * 100
# High Safety: start salary is large relative to the earnings spread
clean_df['Safety']   = clean_df['Start'] / clean_df['Spread']
# ascending=False → rank 1 = highest salary
clean_df['Start_Rank']  = clean_df['Start'].rank(ascending=False)
clean_df['Mid_Rank']    = clean_df['Mid'].rank(ascending=False)
# Positive Rank_Change = climbed by mid-career; negative = fell
clean_df['Rank_Change'] = clean_df['Start_Rank'] - clean_df['Mid_Rank']
clean_df['Salary_Band'] = pd.cut(clean_df['Start'], bins=3, labels=['Low', 'Medium', 'High'])
clean_df['is_STEM']     = clean_df['Group'] == 'STEM'
# transform returns a Series aligned to the full DataFrame — no merge needed
clean_df['Group_Avg_Start'] = clean_df.groupby('Group')['Start'].transform('mean')
clean_df['vs_group_avg']    = clean_df['Start'] - clean_df['Group_Avg_Start']

clean_df[[
    'Major', 'Group', 'Spread', 'Growth', 'Safety',
    'Rank_Change', 'Salary_Band', 'vs_group_avg',
]].head(10)

### What to look for in the output

- **Accounting**: Spread of ~$110k — fairly high variability despite being a stable-sounding field.
- **Aerospace Engineering**: High Safety (high start, moderate spread) — a predictable field with a strong floor.
- **Rank_Change**: Some majors will show large positive values — these are the quiet climbers that look unremarkable at graduation but rise significantly by mid-career.

With features built, we now have everything we need to filter, aggregate, sort, and visualise with real meaning behind each number.

## <a id="section-5-selection"></a>5. Selection & Filtering

Before aggregating or sorting, it is worth understanding how pandas exposes different slices of data. The syntax varies by use case — knowing which tool to reach for prevents both bugs and verbose code.

| Method | Best used when |
|---|---|
| `df['col']` | You want a single column as a Series |
| `df[['c1','c2']]` | You want multiple columns as a DataFrame |
| `.loc[row, col]` | You know the *label* (row index value or column name) |
| `.iloc[r, c]` | You know the *integer position*, not the label |
| Boolean mask `df[condition]` | You want rows matching a criterion |
| `&` multi-condition | Two or more conditions — must parenthesise each |
| `.isin([list])` | Membership test against a list of allowed values |
| `.query('expr')` | SQL-like string syntax — most readable for complex conditions |
| `.between(lo, hi)` | Inclusive range filter — cleaner than `(col >= lo) & (col <= hi)` |
| `.str.contains('pat')` | Substring match on a string column |

**A common mistake:** chaining `df['col'][idx]` instead of `df.loc[idx, 'col']`. Both return the same value, but chaining can produce a `SettingWithCopyWarning` when used on the left side of an assignment. Always use `.loc` when your intent is to write to a cell.

In [ ]:
# Single column → Series
print('First 5 majors:')
print(clean_df['Major'].head(5).tolist())

In [ ]:
# Multiple columns → DataFrame
clean_df[['Major', 'Start', 'Mid']].head()

In [ ]:
# Label-based cell — .loc is the preferred form
print('loc[0, Major]:', clean_df.loc[0, 'Major'])

# Integer-position slice — useful when position matters more than label
print('\niloc[0:5, 0:3]:')
clean_df.iloc[0:5, 0:3]

In [ ]:
# Boolean filter — majors with strong starting salaries
print('Majors with Start > $50,000:')
clean_df[clean_df['Start'] > 50000][['Major', 'Start', 'Group']]

In [ ]:
# Multiple conditions — parenthesise each condition before using &
print('STEM majors with Start > $55,000 — the high-floor STEM fields:')
clean_df[(clean_df['Group'] == 'STEM') & (clean_df['Start'] > 55000)][['Major', 'Start']]

In [ ]:
# isin — filter to multiple category values at once
print('STEM or Business majors (first 8):')
display(clean_df[clean_df['Group'].isin(['STEM', 'Business'])][['Major', 'Group']].head(8))

# query — same as the boolean mask above but reads like plain English
print('\nquery: STEM and Start > $50,000:')
clean_df.query('Group == "STEM" and Start > 50000')[['Major', 'Start']]

In [ ]:
# between — inclusive range filter, cleaner than two separate conditions
print('Majors with Start between $40,000 and $60,000 — the middle-ground:')
display(clean_df[clean_df['Start'].between(40000, 60000)][['Major', 'Start']].head(8))

# str.contains — filter by substring in the major name
print('\nAll Engineering majors:')
clean_df[clean_df['Major'].str.contains('Engineering')][['Major', 'Start', 'Mid']]

## <a id="section-6-aggregation"></a>6. Aggregation

Individual data points are noisy. Aggregation removes the noise and exposes the signal.

The salary data spans 50 majors. Any single number — Chemical Engineering's $107k mid-career, Education's $52k — is a data point. The *mean*, *median*, and *standard deviation* across all majors tell you where the typical outcome sits and how widely it varies. Without these, you cannot contextualise a single major's numbers.

### Key statistics and what they reveal here

- **`sum`**: the total across all majors — not very meaningful here, but useful in other datasets (e.g. total revenue across categories).
- **`mean`**: the arithmetic average — pulled upward by outlier high earners. Compare with median.
- **`median`**: the middle value — more robust to outliers. If mean > median, the top earners are pulling the average up.
- **`std`**: standard deviation — how spread out the values are around the mean. High std on `Start` means starting salaries vary widely across majors.
- **`var`**: variance — the square of std. Less interpretable directly, but used in statistical tests.
- **`idxmin` / `idxmax`**: return the *index* of the extreme value — the row number — so we can look up which major it is.
- **`quantile`**: Q1 (25th percentile) and Q3 (75th percentile) define the interquartile range — where the middle 50% of majors sit.
- **`agg`**: apply multiple functions to multiple columns in a single call — the most efficient way to produce a summary table.

In [ ]:
print(f"Mid sum     : ${clean_df['Mid'].sum():,.0f}")
print(f"Start mean  : ${clean_df['Start'].mean():,.0f}")
print(f"Mid median  : ${clean_df['Mid'].median():,.0f}")
print(f"Start std   : ${clean_df['Start'].std():,.0f}")
print(f"Spread var  : {clean_df['Spread'].var():,.0f}")

In [ ]:
print(f"Mid min : ${clean_df['Mid'].min():,.0f}  |  Mid max : ${clean_df['Mid'].max():,.0f}")
print(f"idxmin  : {clean_df['Mid'].idxmin()}  |  idxmax  : {clean_df['Mid'].idxmax()}")

# What are those majors?
print(f"\nLowest mid-career : {clean_df.loc[clean_df['Mid'].idxmin(), 'Major']}")
print(f"Highest mid-career: {clean_df.loc[clean_df['Mid'].idxmax(), 'Major']}")

top_mid_row = clean_df.loc[clean_df['Mid'].idxmax()]
bot_mid_row = clean_df.loc[clean_df['Mid'].idxmin()]

> **Finding 1 & 2 of 8 — Highest and lowest mid-career earners**
> Chemical Engineering leads mid-career earnings; Education sits at the bottom. The gap between them is over $55,000 — roughly half the STEM ceiling. Both extremes are confirmed by the output above.

In [ ]:
print(f"Start Q1: ${clean_df['Start'].quantile(0.25):,.0f}  |  Q3: ${clean_df['Start'].quantile(0.75):,.0f}")
print(f"IQR     : ${clean_df['Start'].quantile(0.75) - clean_df['Start'].quantile(0.25):,.0f}")

In [ ]:
# agg applies multiple functions at once — a one-call summary table
clean_df[['Start', 'Mid', 'Spread']].agg(['mean', 'median', 'std'])

In [ ]:
# describe gives count, mean, std, min, quartiles, max in one block
clean_df[['Start', 'Mid', 'P10', 'P90', 'Spread', 'Growth']].describe()

### What the numbers show

The mean starting salary across all 50 majors is around **$44,000**, but the standard deviation is large — individual majors range from ~$34k (Spanish) to $74k (Physician Assistant). This wide spread means the average is not a useful number for any individual decision.

The mid-career mean is significantly higher than starting, confirming that salaries do grow — but the interesting question is *which majors grow the most*, which is what the GroupBy and Sorting sections below reveal.

## <a id="section-7-groupby"></a>7. GroupBy

The central analytical question of this dataset — **does your degree category matter?** — can only be answered by grouping. GroupBy is the pandas equivalent of an Excel Pivot Table: it splits the data by a categorical column, applies an aggregation function to each group, and returns a summary table.

The three degree groups — STEM, Business, HASS — have very different distributions. GroupBy lets us compare them directly rather than eyeballing individual rows.

### Pattern to remember

```python
df.groupby('Group')  # split
    .mean()          # aggregate
```

Add `numeric_only=True` when the DataFrame contains string columns — otherwise pandas will raise a `TypeError` when trying to average non-numeric values.

### More powerful GroupBy patterns

- `.agg({'col': ['fn1', 'fn2']})` — different functions per column in one call
- `.nlargest(n)` — top-n rows within each group, without sorting the full DataFrame
- `.apply(lambda)` — arbitrary per-group logic when built-in aggregations aren't enough
- `.transform('mean')` — per-group aggregation broadcast back to every row (already used in section 4)

In [ ]:
print('Average salaries by degree group:')
clean_df.groupby('Group').mean(numeric_only=True)[['Start', 'Mid', 'Spread', 'Growth']]

stem_avg_mid = clean_df[clean_df['Group'] == 'STEM']['Mid'].mean()
hass_avg_mid = clean_df[clean_df['Group'] == 'HASS']['Mid'].mean()

stem_avg_growth = clean_df[clean_df['Group'] == 'STEM']['Growth'].mean()
hass_avg_growth = clean_df[clean_df['Group'] == 'HASS']['Growth'].mean()

In [ ]:
print('Median salaries by group — less sensitive to outliers than mean:')
clean_df.groupby('Group').median(numeric_only=True)[['Start', 'Mid', 'Spread']]

In [ ]:
print('Min and max by group — the floor and ceiling per category:')
display(clean_df.groupby('Group').min(numeric_only=True)[['Start', 'Mid']])
display(clean_df.groupby('Group').max(numeric_only=True)[['Start', 'Mid']])
print('\nMajor count per group:')
clean_df.groupby('Group').count()[['Major']]

In [ ]:
# agg with a dict — different aggregations for different columns in one call
print('Start (mean/min/max) and Spread (mean) per group:')
clean_df.groupby('Group').agg({'Start': ['mean', 'min', 'max'], 'Spread': 'mean'})

In [ ]:
# nlargest within groups — top 3 mid-career earners per category
print('Top 3 Mid-Career earners per group:')
display(clean_df.groupby('Group')['Mid'].nlargest(3))

# apply with a lambda — full row access per group
print('\nTop 2 starters per group (full row context):')
clean_df.groupby('Group', group_keys=False).apply(
    lambda grp: grp.nlargest(2, 'Start')
)[['Major', 'Group', 'Start', 'Mid']]

In [ ]:
# transform result already lives in clean_df from section 4 — show it in context
print('Group average vs individual major start salary:')
clean_df[['Major', 'Group', 'Start', 'Group_Avg_Start', 'vs_group_avg']].head(12)

> **Finding 6 of 8 — STEM vs HASS earnings gap**
> STEM out-earns HASS by roughly $20,000 at mid-career on average. But the HASS average is dragged down by large low-paying fields like Education and Religion. Remove those and the gap narrows significantly. Within-group variance is high on both sides — the category you pick matters less than the specific major.

### What the GroupBy tables reveal

STEM leads on every salary metric — starting, mid-career, and P90. But the picture is more nuanced than that:

- **STEM, Business, and HASS all grow at nearly the same rate** — average Growth is 70.40%, 68.54%, and 68.86% respectively. The groups are almost identical on percentage growth; the STEM advantage is in absolute starting salary, not trajectory.
- **Business has the lowest average Spread** — more predictable mid-career outcomes than STEM.
- The *floor* of HASS (Religion, Education at ~$35k start) drags the group average down significantly. Remove those and the HASS average looks very different.

The next section answers the ranking questions directly.

## <a id="section-8-sorting"></a>8. Sorting & Ranking

`sort_values()` is how we turn aggregated data into rankings. It is the most direct way to answer 'which major is best at X?' — and it reveals patterns that group averages hide.

### Why rankings matter more than averages here

GroupBy gave us STEM > Business > HASS on average. But:
- The *top* HASS major (Economics) out-earns most STEM fields at mid-career.
- The *bottom* STEM major (Biology) earns less than many Business fields.
- The *biggest climber* by Rank_Change is not a STEM field at all.

Rankings expose the within-group variance that averages flatten out.

### `nlargest` vs `sort_values`

For extracting the top-n rows, `nlargest(n, col)` is faster than `sort_values(col, ascending=False).head(n)` because it does not sort the entire DataFrame — it uses a partial sort. For small datasets like this one the difference is negligible, but the pattern is worth knowing.

In [ ]:
print('Top 10 by Mid-Career median salary:')
clean_df.sort_values('Mid', ascending=False)[['Major', 'Group', 'Start', 'Mid']].head(10)

In [ ]:
# Multi-column sort — alphabetical by group, then descending by start within each group
print('By Group (asc), then Start (desc) — top earner within each group shown first:')
clean_df.sort_values(['Group', 'Start'], ascending=[True, False])[
    ['Major', 'Group', 'Start']
].head(10)

In [ ]:
print('Lowest spread — the most predictable (safest) majors:')
clean_df.sort_values('Spread')[['Major', 'Group', 'Spread', 'Mid']].head(10)

low_risk_row = clean_df.loc[clean_df['Spread'].idxmin()]

> **Finding 4 of 8 — Lowest earnings risk**
> Nursing has the smallest salary spread of any major. Regulated pay scales mean graduates earn within a tight, predictable band regardless of employer or location. For someone who values certainty over upside, Nursing is the data-driven choice.

In [ ]:
print('nlargest 5 by P90 — the highest earning ceilings:')
display(clean_df.nlargest(5, 'P90')[['Major', 'Group', 'P90']])

print('\nnsmallest 5 by Spread — the tightest salary bands:')
clean_df.nsmallest(5, 'Spread')[['Major', 'Group', 'Spread']]

high_ceil_row = clean_df.loc[clean_df['P90'].idxmax()]

> **Finding 5 of 8 — Highest earning ceiling**
> Economics' P90 beats every engineering major. The top 10% of Economics graduates out-earn the top 10% of every other field — but this comes with the widest spread in the dataset. High ceiling, high floor uncertainty.

In [ ]:
print('Group averages sorted alphabetically (sort_index on the groupby result):')
display(clean_df.groupby('Group').mean(numeric_only=True)[['Start', 'Mid']].sort_index())

print('\nTop 10 rank climbers — majors that rose most between start and mid-career:')
clean_df.sort_values('Rank_Change', ascending=False)[
    ['Major', 'Group', 'Start_Rank', 'Mid_Rank', 'Rank_Change']
].head(10)

top_grow_row = clean_df.loc[clean_df['Growth'].idxmax()]

> **Finding 3 of 8 — Biggest rank climber**
> Journalism tops the rank-change table (15 positions), followed by Philosophy (12.5) and Art History (11). Several HASS majors rise sharply between graduation and mid-career despite modest starting salaries. A low starting salary is not a reliable prediction of long-term trajectory.

### The Rank_Change insight

The rank climbers list is one of the most interesting outputs in this notebook. Majors like **Philosophy**, **International Relations**, and **Economics** start with mediocre salary ranks — they are outpaced at graduation by almost every engineering field — but by mid-career they have overtaken many of them.

This matters because it means the starting salary of a HASS major is *not* a prediction of its long-term trajectory. The conventional wisdom of 'start salary = lifetime prospects' is not supported by this data for a significant subset of majors.

## <a id="section-9-reshaping"></a>9. Reshaping & Pivoting

Sometimes the data is in the right format for storage but the wrong format for answering a specific question. Reshaping changes the *structure* without changing the underlying values.

### The four reshaping tools used here

**`pivot_table`** — cross-tabulation. Rows become one categorical variable (Group), columns become the metric, and the values are aggregated. This is the pandas equivalent of dragging fields into an Excel pivot table.

**`set_index`** — promotes a column to become the row label. Once `Major` is the index, `.loc['Economics']` retrieves that row by name instead of by number. Useful when you want label-based lookups on non-integer keys.

**`.T` (transpose)** — flips rows and columns. After a `groupby().mean()`, the groups are rows and metrics are columns. Transposing puts metrics as rows and groups as columns — sometimes easier to read when comparing groups side by side.

**`melt`** — converts *wide* format (one column per salary type) to *long* format (one row per major/salary type combination). Wide format is natural for storage; long format is required by seaborn's plotting functions and many statistical models.

In [ ]:
print('pivot_table — Mid-Career salary by group (mean, max, min):')
clean_df.pivot_table(values='Mid', index='Group', aggfunc=['mean', 'max', 'min'])

In [ ]:
# set_index makes major names the row labels — enables .loc by name
print('Economics full profile:')
clean_df.set_index('Major').loc['Economics']

In [ ]:
# Transpose puts salary metrics as rows and groups as columns — easier to read left-to-right
print('Group means transposed — metrics as rows, groups as columns:')
clean_df.groupby('Group').mean(numeric_only=True)[['Start', 'Mid', 'Spread', 'Growth']].T

In [ ]:
# melt converts wide to long — each row becomes one (major, salary type, amount) record
# This is the format seaborn needs for a grouped bar chart
melted = clean_df.melt(
    id_vars=['Major', 'Group'],
    value_vars=['Start', 'Mid', 'P10', 'P90'],
    var_name='Salary_Type',
    value_name='Amount',
)
print(f'Melted shape: {melted.shape}  (was {clean_df.shape[0]} rows × 4 salary cols = {clean_df.shape[0]*4} rows)')
melted.head(10)

## <a id="section-10-apply"></a>10. Apply & Map

Sometimes what you need cannot be expressed as a built-in pandas aggregation. That is where `apply`, `map`, and string accessors come in — they let you bring arbitrary Python logic into the DataFrame.

### When to use each

**`apply(func, axis=1)`** — runs a function row by row, with access to all columns. Flexible, but slower than vectorised operations. Use it when you need conditional logic that depends on multiple columns at once.

**`map(dict)`** — replaces each value in a Series using a dictionary. Fastest way to relabel a categorical column. No loop, no lambda, just a lookup.

**String accessor `.str.*`** — applies string methods element-wise to an entire column. `.str.contains()`, `.str.upper()`, `.str.len()`, `.str.split()` work on the whole column in one call with no loop.

**`pipe(func)`** — passes the DataFrame into a function, enabling clean method chains without intermediate variables. Particularly readable when you want to chain two transformations that would otherwise require a temporary variable.

In [ ]:
# apply — conditional logic that depends on the Mid column value
tier_labels = clean_df.apply(
    lambda row: 'High'   if row['Mid'] > 90000 else
                'Medium' if row['Mid'] > 65000 else 'Low',
    axis=1,
)
print('Salary tier distribution across 50 majors:')
print(tier_labels.value_counts())
print('\nSample — first 10 majors with their tier:')
pd.concat([clean_df[['Major', 'Group', 'Mid']], tier_labels.rename('Tier')], axis=1).head(10)

In [ ]:
# map — replace short codes with full descriptions via a dictionary lookup
group_full = clean_df['Group'].map({
    'STEM':     'Science & Technology',
    'Business': 'Business & Commerce',
    'HASS':     'Humanities & Social Sciences',
})
print('Full group names:')
print(group_full.value_counts())

In [ ]:
# str accessor — element-wise string operations on the Major column
print('str.upper (first 5):')
print(clean_df['Major'].str.upper().head(5).tolist())

print('\nAll Engineering majors (str.contains):')
print(clean_df[clean_df['Major'].str.contains('Engineering')]['Major'].tolist())

print('\nLongest major names (str.len sorted desc):')
print(clean_df['Major'].str.len().sort_values(ascending=False).head(5))

print('\nFirst word of each major (str.split().str[0], first 10):')
print(clean_df['Major'].str.split().str[0].head(10).tolist())

In [ ]:
# pipe — two transformations chained without an intermediate variable
# Reads as: take clean_df, keep only STEM, sort by Mid descending
stem_sorted = (
    clean_df
    .pipe(lambda d: d[d['Group'] == 'STEM'])
    .pipe(lambda d: d.sort_values('Mid', ascending=False))
)
print('STEM majors ranked by Mid-Career salary (pipe):')
stem_sorted[['Major', 'Start', 'Mid']].head(6)

## <a id="section-11-display"></a>11. Display & Styling

Data quality and presentation quality are different things, but both matter. A table of raw floats — `77100.0`, `101000.0` — is harder to read than `$77,100.00` and `$101,000.00`, especially when scanning for differences.

`pd.options.display.float_format` sets a global format string applied to all float output in the notebook. The format `'{:,.2f}'.format` adds comma thousands-separators and two decimal places. It is purely cosmetic — the underlying float values in the DataFrame are unchanged.

`pd.options.display.max_rows` controls truncation. The default is 60, which would hide nothing here (we have 50 majors). Setting it explicitly to 51 ensures the full dataset always displays without the ellipsis interruption, regardless of environment defaults.

We reset both options with `pd.reset_option()` at the end of this section so they do not bleed into subsequent outputs.

In [ ]:
pd.options.display.float_format = '{:,.0f}'.format
pd.options.display.max_rows = 51

print('Top 10 by Mid-Career salary — formatted as currency:')
clean_df.sort_values('Mid', ascending=False)[['Major', 'Group', 'Start', 'Mid']].head(10)

In [ ]:
print('Group summary — average salaries formatted:')
clean_df.groupby('Group').mean(numeric_only=True)[['Start', 'Mid', 'Spread', 'Growth']]

In [ ]:
# Rank change table with a Direction label — Climber / Faller / Stable
rank_table = clean_df[['Major', 'Group', 'Start_Rank', 'Mid_Rank', 'Rank_Change']].copy()
rank_table['Direction'] = rank_table['Rank_Change'].apply(
    lambda x: 'Climber' if x > 0 else ('Faller' if x < 0 else 'Stable')
)
print('Rank change — top 15 climbers:')
rank_table.sort_values('Rank_Change', ascending=False).head(15)

In [ ]:
# Reset so formatting does not affect subsequent sections
pd.reset_option('display.float_format')

## <a id="section-12-correlation"></a>12. Correlation Analysis

Correlation tells us whether two variables move together. A value of **+1** means perfect positive correlation — when one goes up, the other always goes up by a proportional amount. **−1** means perfect inverse. **0** means no linear relationship.

### Why correlation matters for this dataset

We have six salary-related variables. Knowing which ones are strongly correlated answers concrete questions:

- **Is starting salary a good predictor of mid-career salary?** If Start and Mid are strongly correlated (r > 0.7), then a major's starting salary is a reliable indicator of long-term prospects. If not, starting salary is a poor proxy and you should look at other metrics.
- **Do high-ceiling majors (P90) also have high spreads?** If P90 and Spread are strongly correlated, it means the majors with the highest upside also have the most variable outcomes — high reward comes with high risk.
- **Does Growth correlate with anything?** If Growth correlates weakly with Start, that suggests high-growth majors are different from high-starting-salary majors — they are not the same thing.

### Reading the heatmap

The correlation matrix is symmetric — the value at row A, column B equals the value at row B, column A. The diagonal is always 1.0 (a variable is perfectly correlated with itself). We use the upper triangle only to avoid counting each pair twice.

In [ ]:
corr_cols   = ['Start', 'Mid', 'P10', 'P90', 'Spread', 'Growth']
corr_matrix = clean_df[corr_cols].corr()
corr_matrix

In [ ]:
pearson_start_mid = corr_matrix.loc['Start', 'Mid']
print(f'Pearson r (Start vs Mid): {pearson_start_mid:.3f}')
if pearson_start_mid > 0.7:
    print('→ Strong positive. Starting salary IS a meaningful predictor of mid-career salary.')
else:
    print('→ Moderate. Starting salary is only a partial predictor of mid-career salary.')

> **Finding 7 of 8 — Starting salary predicts mid-career salary**
> Pearson r > 0.8 between Start and Mid across the full dataset. Choosing a well-paid field at graduation is not just a short-term win — it predicts the long-term trajectory. The correlation is strong but imperfect: rank climbers like Philosophy and Economics are the exceptions.

In [ ]:
# Upper-triangle mask — avoids counting each pair twice
upper_mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
corr_pairs = (
    corr_matrix.where(upper_mask)
    .stack()
    .sort_values(ascending=False)
)
print('Two most correlated pairs across all salary metrics:')
print(corr_pairs.head(2))

In [ ]:
# A concrete cross-group question: which HASS majors beat the average STEM starting salary?
stem_avg_start    = clean_df[clean_df['Group'] == 'STEM']['Start'].mean()
hass_beating_stem = clean_df[
    (clean_df['Group'] == 'HASS') & (clean_df['Start'] > stem_avg_start)
]
print(f'Average STEM starting salary: ${stem_avg_start:,.0f}')
print(f'\nHASS majors that beat it:')
hass_beating_stem[['Major', 'Start', 'Mid', 'Growth']]

### What the correlation tells us

- **Start ↔ P10**: very high — majors with strong starting salaries also have a high floor at mid-career. A strong start protects against a bad outcome.
- **P90 ↔ Spread**: high — fields with the highest ceilings are also the most variable. Economics, Finance, and Math can make you very rich or just comfortable.
- **Growth ↔ Start**: low or negative — the biggest growers often start below average. This is the hidden argument for some HASS majors: low start, high trajectory.

The HASS majors that beat the average STEM starting salary are a small but important group — they challenge the simple STEM > HASS framing.

## <a id="section-13-export"></a>13. Export

The cleaned and enriched DataFrame — with seven derived columns added — should be saved so downstream consumers (reporting tools, other scripts, dashboards) do not need to re-run the analysis.

Three formats are exported:

- **`clean_salaries.csv`** — the full enriched dataset as a flat file. `index=False` prevents pandas writing a spurious integer index column that would confuse any tool that imports it.
- **`group_summary.csv`** — the groupby mean table: a compact pivot-style summary ready for a report or slide.
- **`salaries.json`** — one JSON object per major (`orient='records'`). This format is immediately consumable by JavaScript, REST APIs, or any tool that prefers JSON over CSV.

In [ ]:
clean_df.to_csv('data/clean_salaries.csv', index=False)
print('Saved: data/clean_salaries.csv')

clean_df.groupby('Group').mean(numeric_only=True).to_csv('data/group_summary.csv')
print('Saved: data/group_summary.csv')

clean_df.to_json('data/salaries.json', orient='records', indent=2)
print('Saved: data/salaries.json')

## <a id="section-14-visualisations"></a>14. Visualisations

Tables answer precise questions. Charts reveal patterns, outliers, and relationships that are hard to see in rows of numbers.

Seven charts are produced, each designed to answer a different question visually:

| Chart | Question answered |
|---|---|
| 01 — Top 10 mid-career | Which specific majors pay the most at mid-career? |
| 02 — Group comparison | How do STEM, Business, HASS compare across all four salary metrics? |
| 03 — Scatter trajectory | Does starting salary predict mid-career salary? Who are the outliers? |
| 04 — Top 10 growth | Which majors grow the most from start to mid-career? |
| 05 — Correlation heatmap | Which salary metrics are most strongly related to each other? |
| 06 — Box plot by group | How spread out are mid-career salaries within each degree group? |
| 07 — Rank change | Which majors climb or fall most between graduation and mid-career? |

**Colour convention used throughout:**  
STEM = `#2E75B6` (blue) · Business = `#ED7D31` (orange) · HASS = `#70AD47` (green)

All charts are saved to `plots/` at 150 dpi and displayed inline.

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
# Chart 1: top 10 mid-career — horizontal bar, coloured by Group
# Horizontal bars work better than vertical when major names are long
top10_mid   = clean_df.sort_values('Mid', ascending=False).head(10)
bar_colours = [GROUP_COLOURS[g] for g in top10_mid['Group']]
fig, ax     = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10_mid['Major'], top10_mid['Mid'], color=bar_colours)
ax.invert_yaxis()  # highest salary at top
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, top10_mid['Mid']):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}', va='center', fontsize=9)
legend_handles = [
    Patch(color=col, label=grp) for grp, col in GROUP_COLOURS.items()
    if grp in top10_mid['Group'].values
]
ax.legend(handles=legend_handles, loc='lower right')
ax.set_title('Top 10 Majors by Mid-Career Median Salary')
ax.set_xlabel('Mid-Career Median Salary')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_top10_midcareer.png', dpi=150)
plt.show()

In [ ]:
# Chart 2: grouped bar — average of each salary metric per degree group
# Shows STEM/Business/HASS side by side across Start, Mid, P10, P90
groups      = ['STEM', 'Business', 'HASS']
metrics     = ['Start', 'Mid', 'P10', 'P90']
group_means = clean_df.groupby('Group')[metrics].mean()
x           = np.arange(len(metrics))
bar_width   = 0.25
fig, ax     = plt.subplots(figsize=(11, 6))
for i, grp in enumerate(groups):
    ax.bar(x + i * bar_width, group_means.loc[grp, metrics],
           width=bar_width, label=grp, color=GROUP_COLOURS[grp])
ax.set_xticks(x + bar_width)
ax.set_xticklabels(metrics)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:,.0f}'))
ax.set_title('Average Salary by Degree Group')
ax.set_xlabel('Salary Metric')
ax.set_ylabel('Average Salary')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_group_comparison.png', dpi=150)
plt.show()

In [ ]:
# Chart 3: scatter — Starting vs Mid-Career, regression line per group, 5 outliers labelled
# The slope and scatter of each group's regression line shows how predictable the trajectory is
fig, ax = plt.subplots(figsize=(10, 7))
for grp in groups:
    subset = clean_df[clean_df['Group'] == grp]
    ax.scatter(subset['Start'], subset['Mid'],
               color=GROUP_COLOURS[grp], label=grp, s=60, alpha=0.8)
    coeffs = np.polyfit(subset['Start'], subset['Mid'], 1)
    x_line = np.linspace(subset['Start'].min(), subset['Start'].max(), 100)
    ax.plot(x_line, np.polyval(coeffs, x_line),
            color=GROUP_COLOURS[grp], linewidth=1.5, linestyle='--', alpha=0.6)

# Outliers = furthest above the overall regression line (mid-career over-performers)
overall_coeffs   = np.polyfit(clean_df['Start'], clean_df['Mid'], 1)
clean_df['_res'] = clean_df['Mid'] - np.polyval(overall_coeffs, clean_df['Start'])
for _, row in clean_df.nlargest(5, '_res').iterrows():
    ax.annotate(row['Major'], (row['Start'], row['Mid']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
clean_df.drop(columns=['_res'], inplace=True)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:,.0f}'))
ax.set_title('Starting vs Mid-Career Salary by Major')
ax.set_xlabel('Starting Median Salary')
ax.set_ylabel('Mid-Career Median Salary')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_scatter_salary_trajectory.png', dpi=150)
plt.show()

In [ ]:
# Chart 4: top 10 growth % — which majors reward long careers most?
top10_growth   = clean_df.sort_values('Growth', ascending=False).head(10)
growth_colours = [GROUP_COLOURS[g] for g in top10_growth['Group']]
fig, ax        = plt.subplots(figsize=(10, 6))
growth_bars    = ax.barh(top10_growth['Major'], top10_growth['Growth'], color=growth_colours)
ax.invert_yaxis()
for bar, val in zip(growth_bars, top10_growth['Growth']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}%', va='center', fontsize=9)
legend_handles = [
    Patch(color=col, label=grp) for grp, col in GROUP_COLOURS.items()
    if grp in top10_growth['Group'].values
]
ax.legend(handles=legend_handles, loc='lower right')
ax.set_title('Top 10 Majors by Salary Growth (Start → Mid-Career)')
ax.set_xlabel('Growth (%)')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_top10_growth.png', dpi=150)
plt.show()

In [ ]:
# Chart 5: correlation heatmap — colour-coded pairwise Pearson coefficients
# coolwarm: blue = negative, white = ~0, red = positive
corr_data = clean_df[['Start', 'Mid', 'P10', 'P90', 'Spread', 'Growth']].corr()
fig, ax   = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_data, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
ax.set_title('Salary Metric Correlation Heatmap')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Chart 6: box plot — distribution of mid-career salaries within each group
# Boxes show Q1–Q3, whiskers extend to 1.5×IQR, dots are outliers
fig, ax     = plt.subplots(figsize=(8, 6))
group_order = ['STEM', 'Business', 'HASS']
group_data  = [clean_df[clean_df['Group'] == g]['Mid'].values for g in group_order]
bp          = ax.boxplot(group_data, patch_artist=True, labels=group_order)
for patch, grp in zip(bp['boxes'], group_order):
    patch.set_facecolor(GROUP_COLOURS[grp])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:,.0f}'))
ax.set_title('Mid-Career Salary Distribution by Degree Group')
ax.set_xlabel('Degree Group')
ax.set_ylabel('Mid-Career Median Salary')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_boxplot_by_group.png', dpi=150)
plt.show()

In [ ]:
# Chart 7: rank change — top 10 climbers (dark green) and top 10 fallers (red)
# Both shown on one chart separated by the zero line
climbers   = clean_df.sort_values('Rank_Change', ascending=False).head(10)
fallers    = clean_df.sort_values('Rank_Change').head(10)
chart_data = (
    pd.concat([climbers, fallers])
    .drop_duplicates('Major')
    .sort_values('Rank_Change')
)
bar_colours_rank = [
    '#C00000' if v < 0 else '#375623' for v in chart_data['Rank_Change']
]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(chart_data['Major'], chart_data['Rank_Change'], color=bar_colours_rank)
ax.axvline(0, color='black', linewidth=0.8)  # zero reference line
ax.set_title('Salary Rank Change: Starting → Mid-Career')
ax.set_xlabel('Rank Change (positive = climbed by mid-career)')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_rank_change.png', dpi=150)
plt.show()

## <a id="section-15-key-findings"></a>15. Key Findings


---

In [ ]:
# pearson_start_mid, stem_avg_start, hass_beating_stem, top_mid_row, bot_mid_row etc. all computed in earlier sections

pd.DataFrame([
    ("Highest mid-career salary",         top_mid_row['Major'],                              f"${top_mid_row['Mid']:,.0f}"),
    ("Lowest mid-career salary",           bot_mid_row['Major'],                              f"${bot_mid_row['Mid']:,.0f}"),
    ("Biggest salary grower",              top_grow_row['Major'],                             f"{top_grow_row['Growth']:.2f}%"),
    ("Lowest earnings risk",               low_risk_row['Major'],                             f"${low_risk_row['Spread']:,.0f} spread"),
    ("Highest earning ceiling (P90)",      high_ceil_row['Major'],                            f"${high_ceil_row['P90']:,.0f}"),
    ("Start → Mid correlation",            "Pearson r",                                       f"{pearson_start_mid:.3f}"),
    ("STEM vs HASS mid-career gap",        "Average difference",                              f"${stem_avg_mid - hass_avg_mid:,.0f}"),
    ("HASS vs STEM avg Growth",             "Average difference",                           f"{hass_avg_growth:.2f}% vs {stem_avg_growth:.2f}% STEM"),
], columns=['Metric', 'Major / Value', 'Figure'])

### 1. Chemical Engineering graduates earn the most at mid-career
Median mid-career salary of **$107,000**. One of only three majors to break $100k at mid-career median, alongside Computer Engineering ($105k) and Electrical Engineering ($103k). All three are STEM — but not all STEM majors approach these numbers. Biology, for example, sits at $64k mid-career — below many Business fields.

### 2. Education and Religion are tied for the lowest mid-career salary
Both at approximately **$52,000**. This is roughly half the mid-career salary of the top STEM fields. The data makes the financial trade-off explicit: choosing Education is a conscious decision to accept lower financial returns in exchange for other rewards. The numbers do not judge that — but they do make it visible.

### 3. Math is the biggest salary grower
Growth of **103.52%** from start to mid-career — the highest of any major in the dataset. Math starts modestly but more than doubles by mid-career, demonstrating that starting salary is a poor proxy for long-term earnings trajectory. Several other HASS majors (Philosophy, International Relations) are close behind, which challenges the assumption that only STEM fields compound well over a career.

### 4. Nursing offers the lowest earnings risk
Spread of **$50,700** — by far the tightest salary band in the dataset. Regulated pay scales in healthcare mean that Nursing graduates earn within a predictable range regardless of employer or location. For someone who values financial stability over upside, Nursing is the data-driven choice.

### 5. Economics has the highest earning ceiling — beating every engineering field
P90 of **$210,000**. The top 10% of Economics graduates out-earn the top 10% of every other major, including Chemical Engineering ($194k) and Finance ($195k). This is not an average — it represents the realistic upside for graduates who enter finance, consulting, or senior management. It comes with the highest Spread in the dataset: the floor is low, the ceiling is sky-high.

### 6. STEM out-earns HASS by roughly $20,000 at mid-career on average
The gap is real — but it is not the whole story. The HASS average is pulled down significantly by large, low-paying fields like Education, Religion, and Sociology. Remove those, and the HASS average rises sharply. The STEM advantage is real for the average graduate, but the within-group variance on both sides is high enough that major choice within a category matters far more than category choice.

### 7. Starting salary is a strong predictor of mid-career salary — but not for everyone
Pearson r > 0.8 between Start and Mid across the full dataset. Majors with high starting salaries generally maintain their position by mid-career. However, the **rank climbers** — Philosophy, International Relations, Economics — show that a below-average starting salary does not guarantee a below-average trajectory. The correlation is real but imperfect.

### 8. HASS and STEM have nearly identical salary growth rates
Average Growth is **68.86%** (HASS) vs **70.40%** (STEM) — a gap of just 1.53 percentage points. The STEM advantage is in absolute starting salary, not in growth trajectory. A HASS major that starts at $40k and grows at the same rate as a STEM major starting at $60k will always earn less in absolute terms — but the proportional trajectory is nearly the same. Choosing STEM purely for the growth rate is not supported by this data.